In [2]:
from dotenv import load_dotenv
import os
from rich import print

load_dotenv(override=True)
API_HOST = os.getenv("API_HOST", "github")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
        model_name=os.getenv("GITHUB_MODEL", "openai/gpt-4o"),
        openai_api_base="https://models.github.ai/inference",
        openai_api_key=os.environ["GITHUB_TOKEN"],
    )

Pydantic Model

In [3]:
from pydantic import BaseModel, Field

# Output of the LLM model
class Movie(BaseModel):
    title:str=Field(description="title of movie")
    year:int=Field(description="year of the movie was released")
    director:str=Field(description="name of the director")
    rating:float=Field(description="the rating of the movie out of 10")

In [4]:
model_with_structure = llm.with_structured_output(schema=Movie)
model_with_structure


RunnableBinding(bound=ChatOpenAI(profile={'max_input_tokens': 128000, 'max_output_tokens': 16384, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x7feb8d193640>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x7feb688466b0>, root_client=<openai.OpenAI object at 0x7feb8d193af0>, root_async_client=<openai.AsyncOpenAI object at 0x7feb68845ea0>, model_name='gpt-4o-mini', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://models.github.ai/inference'), kwargs={'response_format': <class '__main__.Movie'>, 'ls_structured_output_format': {'kwargs': {'method': 'jso

In [8]:
response = model_with_structure.invoke("provide details of movie Avatar")
print(response)

Movie(title='Avatar', year=2009, director='James Cameron', rating=8.0)

Messaged Parsed

In [12]:
from pydantic import BaseModel, Field

# Output of the LLM model
class Movie(BaseModel):
    title:str=Field(...,description="title of movie")
    year:int=Field(...,description="year of the movie was released")
    director:str=Field(...,description="name of the director")
    rating:float=Field(...,description="the rating of the movie out of 10")

model_with_structure = llm.with_structured_output(schema=Movie, include_raw=True)
response = model_with_structure.invoke("provide details of movie Avatar")
print(response)

{
    'raw': AIMessage(
        content='{"director":"James Cameron","rating":7.8,"title":"Avatar","year":2009}',
        additional_kwargs={
            'parsed': Movie(title='Avatar', year=2009, director='James Cameron', rating=7.8),
            'refusal': None
        },
        response_metadata={
            'token_usage': {
                'completion_tokens': 22,
                'prompt_tokens': 115,
                'total_tokens': 137,
                'completion_tokens_details': {
                    'accepted_prediction_tokens': 0,
                    'audio_tokens': 0,
                    'reasoning_tokens': 0,
                    'rejected_prediction_tokens': 0
                },
                'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}
            },
            'model_provider': 'openai',
            'model_name': 'gpt-4o-mini-2024-07-18',
            'system_fingerprint': 'fp_f97eff32c5',
            'id': 'chatcmpl-CtADbk1vvRyHS0m0xWpbeEkaLeA8R',
            'finish_reason': 'stop',
            'logprobs': None
        },
        id='lc_run--019b791a-bf70-7d12-b9c2-8f6c08ab20cd-0',
        usage_metadata={
            'input_tokens': 115,
            'output_tokens': 22,
            'total_tokens': 137,
            'input_token_details': {'audio': 0, 'cache_read': 0},
            'output_token_details': {'audio': 0, 'reasoning': 0}
        }
    ),
    'parsed': Movie(title='Avatar', year=2009, director='James Cameron', rating=7.8),
    'parsing_error': None
}

Nested Structure

In [14]:
class Actor(BaseModel):
    name:str=Field(description="name of the actor")
    role:str

class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(None,description="budget of the film")


In [15]:
model_with_structure = llm.with_structured_output(schema=MovieDetails)
response = model_with_structure.invoke("provide details of movie Avatar")
print(response)

MovieDetails(
    title='Avatar',
    year=2009,
    cast=[
        Actor(name='Sam Worthington', role='Jake Sully'),
        Actor(name='Zoe Saldana', role='Neytiri'),
        Actor(name='Sigourney Weaver', role='Dr. Grace Augustine'),
        Actor(name='Stephen Lang', role='Colonel Miles Quaritch'),
        Actor(name='Michelle Rodriguez', role='Trudy Chacón'),
        Actor(name='Giovanni Ribisi', role='Parker Selfridge'),
        Actor(name='Wes Studi', role='Eytukan'),
        Actor(name='CCH Pounder', role="Mo'at")
    ],
    genres=['Action', 'Adventure', 'Sci-Fi'],
    budget=237000000.0
)

TypeDict

In [18]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    title:Annotated[str,...,"title of the movie"]
    year:Annotated[str,...,"year of the movie"]
    director:Annotated[str,...,"director of the movie"]
    rating:Annotated[float,...,"rating of the movie"]

In [19]:
model_with_typedict = llm.with_structured_output(schema=MovieDict)
response = model_with_typedict.invoke("provide details of movie Avatar")
print(response)

{'title': 'Avatar', 'director': 'James Cameron', 'year': '2009', 'rating': 7.8}

TypeDict with Nested

In [21]:
class Actor(TypedDict):
    name:str=Field(description="name of the actor")
    role:str

class MovieDetails(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(None,description="budget of the film")

model_with_typedict = llm.with_structured_output(schema=MovieDetails)
response = model_with_typedict.invoke("provide details of movie Avatar")
print(response)

{
    'title': 'Avatar',
    'year': 2009,
    'genres': ['Science Fiction', 'Fantasy', 'Action'],
    'budget': 237000000,
    'cast': [
        {'name': 'Sam Worthington', 'role': 'Jake Sully'},
        {'name': 'Zoe Saldana', 'role': 'Neytiri'},
        {'name': 'Sigourney Weaver', 'role': 'Dr. Grace Augustine'},
        {'name': 'Stephen Lang', 'role': 'Colonel Miles Quaritch'},
        {'name': 'Michelle Rodriguez', 'role': 'Trudy Chacón'},
        {'name': 'Giovanni Ribisi', 'role': 'Parker Selfridge'}
    ]
}

Using this in an Agent

In [26]:
# Create an agent
from langchain.agents import create_agent

agent = create_agent(model=llm,
    system_prompt="you are an helpul assistant",
    response_format=MovieDetails,
    tools=None
)

response = agent.invoke({
                        "messages":
                                {"role": "user", "content":"provide details of movie Avatar"}
                         }
)

print(response['structured_response'])

{
    'title': 'Avatar',
    'year': 2009,
    'cast': [
        {'name': 'Sam Worthington', 'role': 'Jake Sully'},
        {'name': 'Zoe Saldana', 'role': 'Neytiri'},
        {'name': 'Sigourney Weaver', 'role': 'Dr. Grace Augustine'},
        {'name': 'Stephen Lang', 'role': 'Colonel Miles Quaritch'},
        {'name': 'Michelle Rodriguez', 'role': 'Trudy Chacón'},
        {'name': 'Giovanni Ribisi', 'role': 'Parker Selfridge'}
    ],
    'genres': ['Action', 'Adventure', 'Fantasy', 'Science Fiction'],
    'budget': 237000000.0
}

Data Classes

In [27]:
from dataclasses import dataclass

@dataclass
class MovieDataClass():
    title:str
    year:str
    director:str
    rating:float

agent = create_agent(model=llm,
    system_prompt="you are an helpul assistant",
    response_format=MovieDataClass,
    tools=None
)

response = agent.invoke({
                        "messages":
                                {"role": "user", "content":"provide details of movie Avatar"}
                         }
)

print(response['structured_response'])

MovieDataClass(title='Avatar', year='2009', director='James Cameron', rating=7.8)